# 47 — DEBIAS prompt smoke test

Verifies that the updated `_DEBIAS_STEP1_TEMPLATE` and `_DEBIAS_STEP2_TEMPLATE` prompts
cause agents to update their opinions in response to messages and that the update
carries forward across days.

**What this notebook checks**

1. Step-1 reasoning text contains language about updating opinions (qualitative spot-check).
2. At least some agents shift opinion between Day 0 and Day 1 (distribution-level check).
3. Day 2 reasoning references Day 1 position, confirming `own_reasoning` continuity.

**Profile (minimal — fast local run)**

- 5 citizens, 2 days, 1 policy (Carbon Tax)
- Package mode off (`communication_mode='single'`)
- Local Qwen3-8B-4bit on `http://localhost:8080/v1`
- Offline political messages (`political_message_source='offline'`, set `v1`)
- `debias=True`, `thinking=False`, `random_seed=42`

## 1. Imports and local-server ping

In [1]:
import os, sys, random, logging
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s', force=True)
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('openai').setLevel(logging.WARNING)
logging.getLogger('httpcore').setLevel(logging.WARNING)

import pandas as pd

from cag.io.llm import ping_local

LOCAL_BASE_URL = 'http://localhost:8080/v1'
LOCAL_MODEL    = 'mlx-community/Qwen3-8B-4bit'

info = ping_local(base_url=LOCAL_BASE_URL)
print('Local server reachable:', info)

Local server reachable: {'object': 'list', 'data': [{'id': 'mlx-community/Qwen3-8B-4bit', 'object': 'model', 'created': 1783814822}]}


In [2]:
import cag.abm.agent as _agent_module
import inspect

print("agent.py loaded from:", inspect.getfile(_agent_module))
print()
print("=== _DEBIAS_STEP1_TEMPLATE ===")
print(_agent_module._DEBIAS_STEP1_TEMPLATE)


agent.py loaded from: /Users/ipivs/src/Github/Climate-Action-GABM/src/cag/abm/agent.py

=== _DEBIAS_STEP1_TEMPLATE ===
{anti_sycophancy}

Given your demographic profile, political history, and psychological values, what factors would shape your view on the following policy?

{policy_question}

Consider factors that might lead you to SUPPORT this policy AND factors that might lead you to OPPOSE it. Think about your voting history, your values, your life circumstances, and the messages and reflections from today and previous days.Consider that people update their opinions incrementally as new information arrives. Each day brings fresh messages — treat today's messages as new evidence and update your *current* position (not your original Day-0 position) by about 5% in the direction of today's compelling information. Prior updates do not prevent further updates; each day's reasoning should build on the last. If you received multiple messages today that push in opposite directions, weigh th

## 2. Build agents

In [3]:
from cag.abm.agent import SurveyedCitizen
from cag.abm.environment import SurveyedNation
from cag.io.survey import load

from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)

N_CITIZENS  = 5
RANDOM_SEED = 42
year = 2026
random.seed(RANDOM_SEED)

UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place='UK',
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap,
    selfenh_map=SelfenhMap,
    openness_map=OpennessMap,
    conformtrad_map=ConformTradMap,
    sdo_map=SDOMap,
    edo_map=EDOMap,
    rwa_map=RWAMap,
)

data = load('../data/yougov_survey_data/YouGovProcessedData.csv')
data = data.sample(n=N_CITIZENS, random_state=RANDOM_SEED).reset_index(drop=True)

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

print(f'Citizens loaded: {len(sn.agents_active)}')

2026-07-12 01:07:04,487 [INFO] Columns with NaN counts (before filtering):
tprofile_gross_household    398
Political_Left_Right          6
dtype: int64
2026-07-12 01:07:04,487 [INFO] Column with most NaNs: tprofile_gross_household (398 NaNs)
2026-07-12 01:07:04,492 [INFO] 1483 rows after filtering.


Citizens loaded: 5


## 3. Simulation config

Single-policy, 2 days, `debias=True`.  All other keys match research-canon defaults.

In [4]:
from cag.abm.sim import SIM_CONFIG
from cag.abm.attributes.opinion import ClimatePolicyID

POLICY = ClimatePolicyID.CARBON_TAX

config = dict(SIM_CONFIG)   # start from defaults
config.update({
    'n_citizens'               : N_CITIZENS,
    'days': [
        {'phases': ['P-A', 'P-B', 'C'], 'policy': POLICY},
        {'phases': ['P-A', 'P-B', 'C'], 'policy': POLICY},
    ],
    'k_peers_per_day'          : 2,
    'communication_mode'       : 'single',
    'llm_model'                : LOCAL_MODEL,
    'llm_provider'             : 'local',
    'local_base_url'           : LOCAL_BASE_URL,
    'thinking'                 : False,
    'debias'                   : True,     # <-- exercises our new prompts
    'political_message_source' : 'offline',
    'political_message_set'    : 'v1',
    'day0_anchor'              : 'ground_truth_with_rationale',
    'random_seed'              : RANDOM_SEED,
})

print(f'Policy          : {POLICY}')
print(f'Days            : {len(config["days"])}')
print(f'debias          : {config["debias"]}')
print(f'LLM             : {config["llm_provider"]}/{config["llm_model"]}')

Policy          : ClimatePolicyID(5)
Days            : 2
debias          : True
LLM             : local/mlx-community/Qwen3-8B-4bit


## 4. Run simulation

In [5]:
import time
from cag.abm.sim import run_simulation

t0 = time.perf_counter()
results = run_simulation(config, sn)
wall = time.perf_counter() - t0
print(f'Simulation complete in {wall:.0f}s ({wall/60:.1f} min)')

2026-07-12 01:07:04,542 [INFO] ===============================================================================
2026-07-12 01:07:04,543 [INFO]                             EXPERIMENT CONFIG
2026-07-12 01:07:04,543 [INFO] ===============================================================================
2026-07-12 01:07:04,544 [INFO] [run-shape]  n_citizens=5  days=2  seed=42  communication_mode=single
2026-07-12 01:07:04,544 [INFO] [exposure]   mode=rule_affinity_rank  targets=None  weights=None  audience_cap=None
2026-07-12 01:07:04,545 [INFO] [broadcast]  reach_a=1.00  reach_b=1.00  targeting_a=random  targeting_b=random  message_source=offline  message_set=v1
2026-07-12 01:07:04,545 [INFO] [peer]       k_peers_per_day=2  network_type=stochastic_block  params={p_intra=0.15, p_inter=0.05}
2026-07-12 01:07:04,546 [INFO] [day0]       anchor=ground_truth_with_rationale
2026-07-12 01:07:04,547 [INFO] [memory]     preset=default  verbatim_window_days=2  anchor=on(ttl=none)  own_reasoning=on  da

Simulation complete in 385s (6.4 min)


## 5. Check 1 — Opinion shifts between Day 0 and Day 1

If the prompt is working, at least some agents should change their numeric opinion
after receiving messages on Day 1.  A zero-shift count for *all* agents would suggest
the update instruction is not being followed.

In [6]:
opin = results['opinion_trajectories']
opin_col = 'numeric' if 'numeric' in opin.columns else 'opinion_numeric'

day0 = opin[opin['day'] == 0][['agent_id', opin_col]].rename(columns={opin_col: 'day0'})
day1 = opin[opin['day'] == 1][['agent_id', opin_col]].rename(columns={opin_col: 'day1'})

merged = day0.merge(day1, on='agent_id')
merged['shift'] = merged['day1'] - merged['day0']

print('Opinion shifts Day 0 → Day 1:')
print(merged[['agent_id', 'day0', 'day1', 'shift']].to_string(index=False))
print()
n_shifted = (merged['shift'] != 0).sum()
print(f'Agents that shifted: {n_shifted} / {len(merged)}')
if n_shifted == 0:
    print('WARNING: no agent shifted opinion — review Step-1 reasoning in section 6.')

Opinion shifts Day 0 → Day 1:
 agent_id  day0  day1  shift
    165.0     1     1      0
    582.0     0     1      1
   1379.0     0     1      1
    713.0    -2     2      4
   1878.0     2     2      0

Agents that shifted: 3 / 5


In [7]:

# Full cumulative opinion trajectory across all days
pivot = opin.pivot(index='agent_id', columns='day', values=opin_col)
pivot.columns = [f'day{c}' for c in pivot.columns]

# Day-to-day shifts for every consecutive pair
days_sorted = sorted(opin['day'].unique())
for d0, d1 in zip(days_sorted, days_sorted[1:]):
    pivot[f'shift_{d0}→{d1}'] = pivot[f'day{d1}'] - pivot[f'day{d0}']

print('Full opinion trajectory and cumulative shifts:')
print(pivot.to_string())
print()
for d0, d1 in zip(days_sorted, days_sorted[1:]):
    col = f'shift_{d0}→{d1}'
    n = (pivot[col] != 0).sum()
    print(f'Day {d0}→{d1}: {n}/{len(pivot)} agents shifted')


Full opinion trajectory and cumulative shifts:
          day0  day1  day2  shift_0→1  shift_1→2
agent_id                                        
165.0        1     1     1          0          0
582.0        0     1     1          1          0
713.0       -2     2     2          4          0
1379.0       0     1     1          1          0
1878.0       2     2     1          0         -1

Day 0→1: 3/5 agents shifted
Day 1→2: 1/5 agents shifted


## 6. Check 2 — Spot-check Step-1 reasoning for update language

Print the Day-1 Step-1 reasoning for each agent.  You should see the model explicitly
mentioning updating its opinion in response to the messages it received.

In [8]:
reasoning = results['survey_reasoning']

# Filter to Day 1 and the target policy
r1 = reasoning[(reasoning['day'] == 1) & (reasoning['policy_id'] == str(POLICY))]

if r1.empty:
    print('No Day-1 reasoning found — check policy_id column values:')
    print(reasoning['policy_id'].unique())
else:
    for _, row in r1.iterrows():
        print(f"=== Agent {row['agent_id']} — Day 1 Step-1 reasoning ===")
        print(row['reasoning_text'] if 'reasoning_text' in row else row.iloc[-1])
        print()

=== Agent 165.0 — Day 1 Step-1 reasoning ===
I am cautiously supportive of a carbon fee and dividend policy. While I remain wary of the administrative complexity and potential economic disruption, the idea of using government resources to benefit the public aligns with my values of fairness and equal opportunity. The recent messages have made me more open to the policy, especially if it is implemented with clear support for affected industries and a focus on balancing environmental responsibility with economic stability.

=== Agent 582.0 — Day 1 Step-1 reasoning ===
I am slightly more supportive of the carbon fee and dividend policy now, but remain cautious. The idea of returning revenue to the public aligns with my belief in equal opportunities, and the potential for economic relief for lower-income households is appealing. However, I am still concerned about the complexity of implementation and its potential unintended consequences on different sectors of the economy.

=== Agent 1379

## 7. Check 3 — Day-2 reasoning references Day-1 position (`own_reasoning` continuity)

Day-2 Step-1 reasoning should mention a previously held position, confirming the
`own_reasoning` section of `assemble_context` is carrying Day-1 survey reasoning forward.

In [9]:
r2 = reasoning[(reasoning['day'] == 2) & (reasoning['policy_id'] == str(POLICY))]

if r2.empty:
    print('No Day-2 reasoning rows — only 2 days were run (Day 0 + Day 1). That is expected.')
    print('Re-run with 3 days in config["days"] to exercise this check.')
else:
    for _, row in r2.iterrows():
        print(f"=== Agent {row['agent_id']} — Day 2 Step-1 reasoning ===")
        print(row['reasoning_text'] if 'reasoning_text' in row else row.iloc[-1])
        print()

=== Agent 165.0 — Day 2 Step-1 reasoning ===
I am cautiously supportive of the carbon fee and dividend policy, but remain cautious. On one hand, the idea of using government resources to benefit the public aligns with my values of fairness and equal opportunity, and the policy seems to balance environmental responsibility with economic stability. On the other hand, I am still wary of the administrative complexity and potential economic disruption, especially for local industries and traditional sectors. I believe the policy could work if implemented carefully with clear safeguards and support for affected communities.

=== Agent 582.0 — Day 2 Step-1 reasoning ===
I am slightly more supportive of the carbon fee and dividend policy now, but remain cautious. The idea of returning revenue to the public aligns with my belief in equal opportunities, and the potential for economic relief for lower-income households is appealing. However, I am still concerned about the complexity of implementa

## 8. Summary

- **Section 5:** opinion shift table — should show non-zero shifts for at least some agents.
- **Section 6:** reasoning text — should contain language like "update", "in light of", "compelling", or similar.
- **Section 7:** Day-2 check (only relevant if you re-run with 3 days).

If all agents show zero shift and the reasoning does not mention updating, the prompt change
may not be reaching the model correctly — re-check `_DEBIAS_STEP1_TEMPLATE` in `agent.py`.